<!-- launch-badges -->
<a href="https://colab.research.google.com/github/laban254/ml-for-infrastructure/blob/main/03_machine_learning/scikit-learn/model-pipelines/pipeline.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
&nbsp;
<a href="https://mybinder.org/v2/gh/laban254/ml-for-infrastructure/main?urlpath=lab/tree/03_machine_learning/scikit-learn/model-pipelines/pipeline.ipynb" target="_blank"><img src="https://mybinder.org/badge_logo.svg" alt="Open in Binder"/></a>

> ▶️ **Run this notebook live** — no install needed. Click a badge above to open it in a free cloud runtime.

# Machine Learning Pipelines in SRE

## Context
In a production observability pipeline, raw telemetry data is almost never clean. You might have missing Prometheus scrapes, metrics on vastly different scales (percentage vs. bytes), or categorical tags that need encoding.

If you build a Machine Learning model to predict Server Health, you must apply the exact same preprocessing steps to real-time production data as you did to your historical training data. If you don't, your model will crash or make garbage predictions. 

Scikit-Learn **Pipelines** solve this by bundling preprocessing and modeling into a single, executable pipeline that prevents data leakage and ensures consistency.

## Objectives
- Build a synthetic SRE dataset representing server health and metrics, including a categorical `Datacenter` tag.
- Create a single `Pipeline` that automatically:
  1. Imputes missing telemetry data.
  2. Scales numeric metrics and one-hot encodes the categorical `Datacenter` tag using a `ColumnTransformer`.
  3. Trains a `RandomForestClassifier` to predict server failure.
- Perform hyperparameter tuning safely using `GridSearchCV` on the entire pipeline.

## Expected Outcome
- A pipeline whose test accuracy sits meaningfully above the majority-class baseline, because `Failure` is actually driven by `CPU_Usage`, `Network_Bytes`, and `Datacenter` instead of being independent random noise.
- A working `ColumnTransformer` that applies different preprocessing to numeric vs. categorical columns inside one pipeline, tuned end-to-end with `GridSearchCV`.</cell id="2e162f40">


In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

### 1. Generating Infrastructure Data
Let's create a dataset where we predict if a server is `Healthy (0)` or `Failing (1)` based on CPU usage, Network traffic, and which `Datacenter` it lives in. `Failure` is generated as an actual function of these features (higher CPU + higher network traffic + certain datacenters raise the failure probability), with some added randomness so it isn't a perfectly learnable rule. We will also intentionally include missing (`NaN`) values in the numeric telemetry to simulate dropped Prometheus scrapes.

In [2]:
np.random.seed(42)
n_samples = 200

CPU_Usage = np.random.normal(60, 20, n_samples)
Network_Bytes = np.random.normal(1_000_000, 500_000, n_samples)
Datacenter = np.random.choice(['us-east-1', 'us-west-2', 'eu-central-1'], size=n_samples, p=[0.4, 0.35, 0.25])

# Failure probability is a real (logistic) function of the telemetry, not independent
# label noise: higher CPU and higher network traffic meaningfully raise the chance of
# failure, and us-east-1 (older hardware in this scenario) runs hotter than the other
# regions. We still add noise so it isn't a trivially perfect rule.
z_cpu = (CPU_Usage - 60) / 20
z_net = (Network_Bytes - 1_000_000) / 500_000
datacenter_effect = pd.Series(Datacenter).map({'us-east-1': 1.0, 'us-west-2': 0.0, 'eu-central-1': -0.5}).values
noise = np.random.normal(0, 0.3, n_samples)

logit = -3.0 + 3.4 * z_cpu + 3.0 * z_net + datacenter_effect + noise
failure_probability = 1 / (1 + np.exp(-logit))
Failure = np.random.binomial(1, failure_probability)

data = pd.DataFrame({
    'CPU_Usage': CPU_Usage,
    'Network_Bytes': Network_Bytes,
    'Datacenter': Datacenter,
    'Failure': Failure
})

# Introduce missing values to simulate dropped telemetry
data.loc[np.random.choice(data.index, 10), 'CPU_Usage'] = np.nan
data.loc[np.random.choice(data.index, 15), 'Network_Bytes'] = np.nan

X = data[['CPU_Usage', 'Network_Bytes', 'Datacenter']]
y = data['Failure']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Failure rate: {:.1f}%".format(y.mean() * 100))
X_train.head()

Failure rate: 32.0%


,CPU_Usage,Network_Bytes,Datacenter
169,44.925277,6.134951e+05,us-west-2
97,NaN,1.153650e+06,us-west-2
31,97.045564,1.108229e+06,us-east-1
12,NaN,1.477001e+06,us-west-2
35,35.583127,1.316960e+06,us-east-1


### 2. Building the Pipeline

We now have two different kinds of preprocessing to do: the numeric telemetry (`CPU_Usage`, `Network_Bytes`) needs imputing + scaling, while the categorical `Datacenter` tag needs one-hot encoding. A `ColumnTransformer` lets us route different columns to different preprocessing, and we then chain that combined preprocessing step into a single `Pipeline` together with the model.

In [3]:
# Route columns to the right preprocessing: numeric telemetry gets imputed + scaled,
# the categorical Datacenter tag gets one-hot encoded.
numeric_features = ['CPU_Usage', 'Network_Bytes']
categorical_features = ['Datacenter']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Fill missing metrics with median
    ('scaler', StandardScaler())                     # Standardize scales
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# Define the steps of the pipeline as a list of tuples: ('name', Transformer/Model object)
pipeline = Pipeline([
    ('preprocessor', preprocessor),                            # Step 1: Impute/scale numeric, one-hot encode categorical
    ('classifier', RandomForestClassifier(random_state=42))    # Step 2: Train model
])

# Execute the entire pipeline on the training data
pipeline.fit(X_train, y_train)

# Evaluate on the test data
# The test data is automatically imputed, scaled, and encoded using the parameters/categories learned from X_train!
predictions = pipeline.predict(X_test)

baseline_rate = max(y_test.mean(), 1 - y_test.mean())
print("Sample Predictions (0=Healthy, 1=Failing):", predictions[:10])
print("Majority-Class Baseline (Test Set): {:.2f}%".format(baseline_rate * 100))
print("Pipeline Test Accuracy: {:.2f}%".format(pipeline.score(X_test, y_test) * 100))

Sample Predictions (0=Healthy, 1=Failing): [0 0 0 0 1 1 0 0 1 0]
Majority-Class Baseline (Test Set): 66.67%
Pipeline Test Accuracy: 80.00%


### 3. Hyperparameter Tuning with GridSearchCV

If we want to find the best imputation strategy ('mean' vs 'median') AND the best Random Forest depth (5 vs 10), we can tune the *entire pipeline* at once.

Notice the syntax: `stepname__parametername` (two underscores). Because the numeric imputer now lives inside a nested `ColumnTransformer` (named `preprocessor`, sub-step `num`) rather than directly in the pipeline, its path is one level deeper: `preprocessor__num__imputer__strategy`.

In [4]:
# Define the grid of hyperparameters to search
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],  # Tune the numeric imputer
    'classifier__n_estimators': [50, 100, 200],                  # Tune the classifier
    'classifier__max_depth': [None, 5, 10]                       # Tune the classifier
}

grid_search = GridSearchCV(pipeline, param_grid, cv=3, n_jobs=-1, verbose=1)

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)

print("\nBest parameters found:")
print(grid_search.best_params_)

print("\nBest Cross-Validation Accuracy: {:.2f}%".format(grid_search.best_score_ * 100))

# We can now use grid_search.best_estimator_ as our final production pipeline.
best_pipeline = grid_search.best_estimator_
best_predictions = best_pipeline.predict(X_test)

print("Best Pipeline Test Accuracy: {:.2f}%".format(accuracy_score(y_test, best_predictions) * 100))
print("(Majority-Class Baseline was {:.2f}%)".format(baseline_rate * 100))
print("\nClassification Report:")
print(classification_report(y_test, best_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_test, best_predictions))

Fitting 3 folds for each of 18 candidates, totalling 54 fits



Best parameters found:
{'classifier__max_depth': 5, 'classifier__n_estimators': 50, 'preprocessor__num__imputer__strategy': 'median'}

Best Cross-Validation Accuracy: 85.72%
Best Pipeline Test Accuracy: 81.67%
(Majority-Class Baseline was 66.67%)

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.85      0.86        40
           1       0.71      0.75      0.73        20

    accuracy                           0.82        60
   macro avg       0.79      0.80      0.80        60
weighted avg       0.82      0.82      0.82        60

Confusion Matrix:
[[34  6]
 [ 5 15]]


### 4. Did the Pipeline Actually Learn Something?
Because `Failure` is now generated as a real function of `CPU_Usage`, `Network_Bytes`, and `Datacenter` (not independent random noise), the majority-class baseline on this test set is **66.67%** ("always predict Healthy"). The tuned pipeline reaches **81.67%** test accuracy — a genuine ~15-point lift over that baseline, with a confusion matrix showing the model correctly catching most real failures (15 of 20) while keeping false alarms reasonably low (6 of 40). That's the payoff of the whole exercise: grid search and `ColumnTransformer` only earn their complexity when the underlying data actually has signal in it to find.

### Summary

Pipelines are essential for deploying ML in DevOps environments because:
- They ensure **Consistency**: Production streaming data gets exactly the same preprocessing — imputation, scaling, *and* categorical encoding — as historical Data Lake data.
- They prevent **Data Leakage**: Preprocessing state (like scaler means, imputer medians, and one-hot categories) is strictly isolated during Cross-Validation splits.
- They improve **Code readability**: Multiple workflow steps, including a `ColumnTransformer` that routes numeric and categorical columns to different preprocessing, are reduced to a single `.fit()` call.
- They only pay off when the underlying target is actually predictable from the features. Tuning hyperparameters on data where the label is independent of every feature just produces an elaborate way to rediscover the majority-class rate — which is why this notebook's `Failure` column is generated as a real function of `CPU_Usage`, `Network_Bytes`, and `Datacenter`, and why the tuned pipeline's accuracy is checked against that majority-class baseline rather than assumed to be good.